# Identificación del Sistema Depredador-Presa usando NN y ANFIS

**Autores:** Emmanuel Guerrero Piza, Kevin Andrés Forero Guaitero  
**Universidad Distrital**  

Segunda parte del proyecto de Cibernética III.  
Se recolectan datos por simulación del modelo Lotka-Volterra y se identifican
modelos predictivos usando Redes Neuronales (NN) y ANFIS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from simulacion import simulate, generar_datos
from identificacion import (
    IdentificadorNN, IdentificadorANFIS,
    metricas, mae, mse, rmse
)

np.random.seed(42)

## 1. Generación de datos

Se simula el modelo Lotka-Volterra para múltiples condiciones iniciales y
parámetros. Se usa semilla fija para reproducibilidad.

In [ ]:
# ~50K muestras: ANFIS usa mini-batch para el GD, todos los datos para LS
N_SECUENCIAS = 150
N_STEPS = 300

X, y, params = generar_datos(
    n_secuencias=N_SECUENCIAS,
    n_steps=N_STEPS,
    variar_params=True
)

split = int(0.8 * X.shape[0])
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Datos totales: {X.shape[0]} muestras')
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 2. Entrenamiento: Red Neuronal

Arquitectura: 2 entradas $(x_k, y_k) \to$ 2 capas ocultas (32, 16) $\to$ 2 salidas $(x_{k+1}, y_{k+1})$.

In [ ]:
id_nn = IdentificadorNN(hidden_layer_sizes=(32, 16), max_iter=3000, random_state=42, alpha=0.001)
id_nn.entrenar(X_train, y_train)

y_pred_nn = id_nn.predecir(X_test)
metrics_nn = metricas(y_test, y_pred_nn)

print('=== NN - Metricas sobre test ===')
for k, v in metrics_nn.items():
    print(f'  {k}: {v:.6f}')
print(f'Epocas usadas: {len(id_nn.model.loss_curve_)}')

## 3. Entrenamiento: ANFIS

Arquitectura: rejilla de 5 funciones de membresía gaussianas fijas por entrada $\to$ 25 reglas difusas $\to$ Ridge regression sobre las fuerzas de disparo normalizadas.

In [ ]:
id_anfis = IdentificadorANFIS(n_mf=5, alpha=1.0, noise_std=0.5)
id_anfis.entrenar(X_train, y_train)

y_pred_anfis = id_anfis.predecir(X_test)
metrics_anfis = metricas(y_test, y_pred_anfis)

print('=== ANFIS - Metricas sobre test ===')
for k, v in metrics_anfis.items():
    print(f'  {k}: {v:.6f}')

## 4. Comparación de métricas

In [ ]:
from tabulate import tabulate

tabla = [
    ['NN', metrics_nn['MAE'], metrics_nn['MSE'], metrics_nn['RMSE']],
    ['ANFIS', metrics_anfis['MAE'], metrics_anfis['MSE'], metrics_anfis['RMSE']],
]
print(tabulate(tabla, headers=['Modelo', 'MAE', 'MSE', 'RMSE'],
               tablefmt='grid', floatfmt='.6f'))

## 5. Predicción de trayectorias

Se comparan las trayectorias predichas por NN y ANFIS contra el modelo original.

In [ ]:
a, b, c, d = 0.3, 0.006, 0.018, 0.7
x0_plot, y0_plot = 40, 50
n_plot = 200

x_real, y_real = simulate(a, b, c, d, n_plot, x0_plot, y0_plot)
x_nn, y_nn = id_nn.predecir_trayectoria(x0_plot, y0_plot, n_plot)
x_anfis, y_anfis = id_anfis.predecir_trayectoria(x0_plot, y0_plot, n_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_real, label='Original', lw=2)
axes[0].plot(x_nn, '--', label='NN', lw=1.5)
axes[0].plot(x_anfis, ':', label='ANFIS', lw=1.5)
axes[0].set_xlabel('Paso discreto k')
axes[0].set_ylabel('Perfiles x(k)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_title('Predicción: Perfiles')

axes[1].plot(y_real, label='Original', lw=2)
axes[1].plot(y_nn, '--', label='NN', lw=1.5)
axes[1].plot(y_anfis, ':', label='ANFIS', lw=1.5)
axes[1].set_xlabel('Paso discreto k')
axes[1].set_ylabel('Usuarios y(k)')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_title('Predicción: Usuarios')

plt.tight_layout()
plt.savefig('prediccion_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Diagrama de fase

In [ ]:
plt.figure(figsize=(6, 5))
plt.plot(x_real, y_real, label='Original', lw=2)
plt.plot(x_nn, y_nn, '--', label='NN', lw=1.5)
plt.plot(x_anfis, y_anfis, ':', label='ANFIS', lw=1.5)
plt.xlabel('Perfiles x(k)')
plt.ylabel('Usuarios y(k)')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Diagrama de fase')
plt.savefig('diagrama_fase.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evolución de la pérdida durante entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(id_nn.model.loss_curve_)
axes[0].set_xlabel('Iteración')
axes[0].set_ylabel('Pérdida')
axes[0].set_title('NN: Evolución de pérdida')
axes[0].grid(alpha=0.3)

axes[1].plot(id_anfis.loss_history)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Error cuadrático medio')
axes[1].set_title('ANFIS: Evolución de error')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evolucion_perdida.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusiones

- La Red Neuronal y ANFIS logran identificar la dinámica del sistema depredador-presa.
- Se compararon las trayectorias predichas contra el modelo original.
- Las métricas MAE, MSE y RMSE cuantifican el error de predicción.
- El uso de semilla fija garantiza reproducibilidad de los resultados.